# P2: Sentiment Klassifikatori — Hissiyot Tahlili
**Mavzu:** Logistik regressiya, Naive Bayes, baholash metrikalari (precision/recall/F1)
**Kun:** 3-kun amaliyoti — 18-iyun 2026, 09:30-10:50 (80 daqiqa)
**Juftlashgan ma'ruza:** L2 — Klassik usullar bilan tasniflash, metrikalar, etika
**Kapstone modul:** m02 — `SentimentClassifier` (`capstone/modules/m02_sentiment_classifier.py`)
**Ma'lumot:** Uzum Market sharhlari (`risqaliyevds/uzbek-sentiment-analysis`, MIT) — binarizatsiya: reyting {4,5}->ijobiy, {1,2}->salbiy.

**Bugungi maqsadlar:**
1. Sharhlarni ijobiy/salbiy ga binarizatsiya qilish va o'quv/sinov qismlariga ajratish
2. TF-IDF ustida LogisticRegression va MultinomialNB modellarini o'qitish
3. Naive Bayes ni qo'lda hisoblab, L2 [I2]-slayd natijasini tekshirish (nisbat=3.375)
4. precision, recall, F1 va confusion matrix bilan modellarni baholash
5. m02 `SentimentClassifier` ni `capstone/contracts.py` imzolariga muvofiq yozish

| Bo'lim | Vaqt |
|--------|------|
| §1 Muhit tekshiruvi | 3 daq |
| §2 Yaxlit natija (destination first) | 8 daq |
| §3 Tayyor kod bloki — PRIMM | 17 daq |
| §4 Asosiy mavzu — so'nuvchi tayanch | 35 daq |
| §5 Kapstone integratsiya | 13 daq |
| §6 Tadqiqot savoli + yakun | 7 daq |


In [ ]:
# ═══════════════════════════════════════════════════════════════
# §1  Muhit tekshiruvi
# ═══════════════════════════════════════════════════════════════
import random, os, sys
from pathlib import Path
import numpy as np

# Takrorlanish uchun urug' (seed) — asserts stabilligini ta'minlaydi
random.seed(42)
np.random.seed(42)

OFFLINE_FALLBACK = True   # <-- bu qiymatni o'zgartirmang

CHECKPOINT_DIR = Path("d03_checkpoints")
assert CHECKPOINT_DIR.exists(), (
    "d03_checkpoints/ papkasi topilmadi. "
    "Notebookni course/practices/ papkasidan ishga tushiring yoki: git pull."
)

# Kapstone modullari yo'li (m01 TextPreprocessor ni import qilish uchun)
REPO_ROOT = Path().resolve().parent.parent
MODULES_DIR = REPO_ROOT / "capstone" / "modules"
if str(MODULES_DIR) not in sys.path:
    sys.path.insert(0, str(MODULES_DIR))

try:
    import torch
    print("GPU mavjud — lekin bu amaliyot CPUda ishlatiladi."
          if torch.cuda.is_available() else "CPU rejimi: to'g'ri!")
except ImportError:
    print("PyTorch o'rnatilmagan — bu amaliyotda kerak emas (CPU, klassik ML).")

import sklearn
print(f"Python      : {sys.version.split()[0]}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"numpy       : {np.__version__}")
print("\n✓ Muhit tayyor.")


---
## §2  Yaxlit Natija — Pirovard Manzil Birinchi! (~8 daqiqa)

Quyida **tugallangan sentiment pipeline** bir necha satrda ishga tushadi:
sharhlar -> tozalash -> TF-IDF -> LogisticRegression -> bashorat.
Hozir tushuntirishdan oldin — natijani ko'ring.


In [ ]:
# Pirovard natija (inline versiya — m02 hali yozilmagan)
from pathlib import Path
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

DATA_FILE = Path("d03_checkpoints/uz_sentiment_mini.txt")

def _binarize(rating):
    r = int(rating)
    return "ijobiy" if r >= 4 else ("salbiy" if r <= 2 else None)

_texts0, _labels0 = [], []
for line in DATA_FILE.open(encoding="utf-8"):
    if not line.strip():
        continue
    rating, text = line.rstrip("\n").split("\t", 1)
    lab = _binarize(rating)
    if lab is not None:
        _texts0.append(text); _labels0.append(lab)

# Mini preprocessing (to'liq versiyasi m01 da)
_APO = re.compile(r"['\u2019\u2018]")
_TOK = re.compile(r"[a-z][a-z']*")
def _prep(t):
    return " ".join(_TOK.findall(_APO.sub("'", t).lower()))

vec0 = TfidfVectorizer()
X0 = vec0.fit_transform([_prep(t) for t in _texts0])
clf0 = LogisticRegression(max_iter=1000).fit(X0, _labels0)

sample = "Mahsulot juda sifatli, tez yetkazib berishdi, rahmat."
pred = clf0.predict(vec0.transform([_prep(sample)]))[0]
print(f"Korpus      : {len(_texts0)} sharh "
      f"({_labels0.count('ijobiy')} ijobiy / {_labels0.count('salbiy')} salbiy)")
print(f"Namuna sharh: '{sample}'")
print(f"Bashorat    : {pred.upper()}")
print("\n✓ Pipeline ishladi! Quyida har qadamni o'rganamiz.")


---
## §3  Tayyor Kod Bloki — PRIMM (~17 daqiqa)

### 3A. Ma'lumotni Yuklash va Binarizatsiya

> **Bashorat qiling** — kodni ishlatishdan oldin javob bering:
> Reyting 1-5 ni ikkilik (ijobiy/salbiy) yorliqqa aylantirsak, reyting **3**
> bilan nima qilamiz? Nega?


In [ ]:
# ─── TO'LIQ BERILGAN KOD (PRIMM — periphery) ─────────────────────────────
# Onlayn: Uzum Market sharhlari (risqaliyevds/uzbek-sentiment-analysis, MIT).
# Offline: bundled kichik ulush (d03_checkpoints/uz_sentiment_mini.txt).
from pathlib import Path

DATA_FILE = Path("d03_checkpoints/uz_sentiment_mini.txt")

def binarize_rating(rating):
    """Reyting 1-5 -> ikkilik yorliq. {4,5}->ijobiy, {1,2}->salbiy, 3 tashlanadi."""
    r = int(rating)
    if r >= 4:
        return "ijobiy"
    if r <= 2:
        return "salbiy"
    return None   # neytral (3) tashlanadi

if OFFLINE_FALLBACK or not DATA_FILE.exists():
    source = "offline (bundled namuna)"
else:
    # Onlayn rejim (Kaggle/HF):
    #   from datasets import load_dataset
    #   ds = load_dataset("risqaliyevds/uzbek-sentiment-analysis")["train"]
    #   raw_rows = [(ex["rating"], ex["text"]) for ex in ds]
    source = "online (HF dataset)"

raw_rows = [
    line.rstrip("\n").split("\t", 1)
    for line in DATA_FILE.open(encoding="utf-8") if line.strip()
]

texts, labels = [], []
n_dropped = 0
for rating, text in raw_rows:
    lab = binarize_rating(rating)
    if lab is None:
        n_dropped += 1
        continue
    texts.append(text); labels.append(lab)

print(f"Manba         : {source}")
print(f"Jami o'qildi  : {len(raw_rows)} qator")
print(f"Tashlandi (3) : {n_dropped} neytral sharh")
print(f"Qoldi         : {len(texts)} sharh  "
      f"(ijobiy: {labels.count('ijobiy')}, salbiy: {labels.count('salbiy')})")
print(f"Namuna        : [{labels[0]}] {texts[0]!r}")


> **Tekshiring:**
> 1. Nechta neytral (reyting 3) sharh tashlandi?
> 2. `binarize_rating(3)` nimani qaytaradi? Nega `None` foydali?
> 3. Sinflar muvozanatli (balanced) mi — ijobiy va salbiy soni teng-mi?

> **O'zgartiring — shaxsiy tajriba!**
> `binarize_rating` ni o'zgartiring: reyting 3 ni ham `ijobiy` deb hisoblang.
> Sinflar balansi qanday o'zgaradi? (Keyin asl holatga qaytaring.)


### 3B. m01 bilan Tozalash, Bo'lish va TF-IDF

> **Bashorat qiling:**
> O'tgan amaliyotda (P1) qurgan `TextPreprocessor` (m01) ni qayta ishlatamiz.
> `train_test_split(..., stratify=labels)` — `stratify` nima uchun kerak?


In [ ]:
# ─── TO'LIQ BERILGAN KOD (PRIMM — periphery) ─────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# m01 (P1 da qurgan modul) ni qayta ishlatamiz — kapstone uzviyligi
from m01_text_preprocessor import TextPreprocessor
pre = TextPreprocessor()

def to_clean_string(text):
    """m01 bilan tozalab, TF-IDF uchun bo'sh-joy bilan birlashtiradi."""
    if not text or not text.strip():
        return ""
    return " ".join(pre.preprocess(text))

X_text = [to_clean_string(t) for t in texts]

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    X_text, labels, test_size=0.25, random_state=42, stratify=labels
)

vec = TfidfVectorizer()
X_train = vec.fit_transform(X_train_txt)
X_test  = vec.transform(X_test_txt)

print(f"O'quv (train): {X_train.shape[0]} sharh    Sinov (test): {X_test.shape[0]} sharh")
print(f"TF-IDF lug'at: {X_train.shape[1]} so'z")
print(f"Train balans : ijobiy={y_train.count('ijobiy')}  salbiy={y_train.count('salbiy')}")
print(f"Test balans  : ijobiy={y_test.count('ijobiy')}  salbiy={y_test.count('salbiy')}")


> **Tekshiring:**
> 1. `X_train` va `X_test` necha qatordan iborat? Yig'indisi jami sharhga tengmi?
> 2. TF-IDF lug'ati (`X_train.shape[1]`) necha so'z? Nega `vec` faqat **train** da `fit` qilinadi?
> 3. `X_test = vec.transform(...)` da nega `fit_transform` emas, `transform`?


In [ ]:
# ─── CHECKPOINT A ─────────────────────────────────────────────────────────
import json
from sklearn.feature_extraction.text import TfidfVectorizer
_ckpt = Path("d03_checkpoints/_ckpt_a_split.json")

if OFFLINE_FALLBACK and _ckpt.exists():
    _d = json.loads(_ckpt.read_text(encoding="utf-8"))
    X_train_txt, X_test_txt = _d["Xtr"], _d["Xte"]
    y_train, y_test = _d["ytr"], _d["yte"]
    vec = TfidfVectorizer()
    X_train = vec.fit_transform(X_train_txt)
    X_test  = vec.transform(X_test_txt)
    print(f"Checkpoint A yuklandi: {len(y_train)} train / {len(y_test)} test")
else:
    _ckpt.write_text(json.dumps(
        {"Xtr": X_train_txt, "Xte": X_test_txt, "ytr": y_train, "yte": y_test},
        ensure_ascii=False), encoding="utf-8")
    print("Checkpoint A saqlandi.")


---
## §4  Asosiy Mavzu — So'nuvchi Tayanch (~35 daqiqa)

Uch bosqichda o'rganamiz:
- **Namuna** (Men ko'rsataman) — to'liq kod + tushuntirish
- **Birgalikda** (Birga to'ldiramiz) — blanklar `# === SIZNING KODINGIZ ===`
- **Mustaqil** (Siz qilasiz) — o'z sharhlaringiz, scaffold yo'q


### 4A. Namuna: Naive Bayes ni Qo'lda Hisoblaymiz — L2 [I2]-Slayd

**Bu katakning natijasi to'g'ridan-to'g'ri L2 ma'ruzasidagi qo'lda misol bilan bog'liq.**
Laplace silliqlash bilan: `P(w|c) = (count(w,c) + 1) / (N_c + |V|)`.


In [ ]:
# ═══ NAMUNA (Men ko'rsataman): Naive Bayes ni qo'lda hisoblaymiz ═══
# Ma'ruza L2 [I2]-slayd: pos=['yaxshi film','ajoyib film'], neg=['yomon film']
from collections import Counter

pos_docs = ["yaxshi film", "ajoyib film"]
neg_docs = ["yomon film"]
vocab = set(w for d in pos_docs + neg_docs for w in d.split())
V = len(vocab)                                  # |V| = 4

P_pos = len(pos_docs) / (len(pos_docs) + len(neg_docs))   # 2/3
P_neg = len(neg_docs) / (len(pos_docs) + len(neg_docs))   # 1/3

pos_counts = Counter(w for d in pos_docs for w in d.split())
neg_counts = Counter(w for d in neg_docs for w in d.split())
N_pos, N_neg = sum(pos_counts.values()), sum(neg_counts.values())   # 4, 2
ALPHA = 1                                        # Laplace silliqlash

def p_word(w, counts, N):
    return (counts.get(w, 0) + ALPHA) / (N + ALPHA * V)

test_tokens = ["yaxshi", "film"]
score_pos, score_neg = P_pos, P_neg
for w in test_tokens:
    score_pos *= p_word(w, pos_counts, N_pos)
    score_neg *= p_word(w, neg_counts, N_neg)

ratio = score_pos / score_neg
result = "ijobiy" if score_pos > score_neg else "salbiy"

print(f"P(film|pos)   = {p_word('film', pos_counts, N_pos):.4f}   "
      f"P(yaxshi|pos) = {p_word('yaxshi', pos_counts, N_pos):.4f}")
print(f"P(film|neg)   = {p_word('film', neg_counts, N_neg):.4f}   "
      f"P(yaxshi|neg) = {p_word('yaxshi', neg_counts, N_neg):.4f}")
print(f"score_pos = {score_pos:.5f}   score_neg = {score_neg:.5f}")
print(f"nisbat (ratio) = {ratio:.3f}   ->   bashorat = {result}")

# ─── ASSERT: Ma'ruza L2 [I2]-slayd bilan solishtiring ─────────────────────
assert abs(ratio - 3.375) < 0.01, (
    f"nisbat={ratio:.4f}, kutilgan 3.375. Laplace formulasini tekshiring."
)
assert result == "ijobiy", f"bashorat={result}, kutilgan 'ijobiy'."
# Ma'ruza L2 [I2]-slayd bilan solishtiring
print("\n✓ Ma'ruza L2 [I2]-slayd tasdiqlandi: nisbat=3.375 -> ijobiy")


In [ ]:
# Qo'lda hisob <-> kutubxona ko'prigi: sklearn MultinomialNB xuddi shu natijani beradimi?
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

cv = CountVectorizer()
Xtiny = cv.fit_transform(pos_docs + neg_docs)
ytiny = ["ijobiy", "ijobiy", "salbiy"]
nb_tiny = MultinomialNB(alpha=1.0).fit(Xtiny, ytiny)

proba = nb_tiny.predict_proba(cv.transform(["yaxshi film"]))[0]
pmap = dict(zip(nb_tiny.classes_, proba))
sk_ratio = pmap["ijobiy"] / pmap["salbiy"]

print(f"sklearn: P(ijobiy)={pmap['ijobiy']:.4f}  P(salbiy)={pmap['salbiy']:.4f}  nisbat={sk_ratio:.3f}")
assert nb_tiny.predict(cv.transform(["yaxshi film"]))[0] == "ijobiy"
assert abs(sk_ratio - 3.375) < 0.05, (
    f"sklearn nisbat={sk_ratio:.3f}, qo'lda hisob 3.375 ga mos kelmadi."
)
print("✓ sklearn MultinomialNB qo'lda hisob bilan mos keldi: nisbat≈3.375")


### 4B. Birgalikda: Logistik Regressiya Modelini O'qitamiz

TF-IDF (`X_train`) ustida `LogisticRegression` ni o'qitamiz.


In [ ]:
# ═══ BIRGALIKDA: Logistik regressiya ═══
from sklearn.linear_model import LogisticRegression

# === SIZNING KODINGIZ (2 qator) ===
# 1. LogisticRegression(max_iter=1000) yarating -> clf_lr
# 2. clf_lr.fit(X_train, y_train) bilan o'qiting
clf_lr = LogisticRegression(max_iter=1000)
clf_lr.fit(X_train, y_train)


In [ ]:
# ─── Tekshirish ────────────────────────────────────────────────────────────
from sklearn.exceptions import NotFittedError
assert clf_lr is not None, "clf_lr None. LogisticRegression() yaratdingizmi?"
try:
    _acc = clf_lr.score(X_train, y_train)
except NotFittedError:
    raise AssertionError("clf_lr o'qitilmagan. clf_lr.fit(X_train, y_train) ni chaqiring.")
assert set(clf_lr.classes_) == {"ijobiy", "salbiy"}, "Sinflar {'ijobiy','salbiy'} bo'lishi kerak."
assert _acc > 0.8, f"O'quv aniqligi past ({_acc:.2f}). fit() to'g'ri ishladimi?"
print(f"✓ LogReg o'qitildi. O'quv aniqligi: {_acc:.3f}   Sinflar: {list(clf_lr.classes_)}")


### 4C. Birgalikda: Naive Bayes Modelini O'qitamiz

`MultinomialNB(alpha=1.0)` — `alpha=1.0` bu Laplace silliqlash (4A dagi `ALPHA`).


In [ ]:
# ═══ BIRGALIKDA: Naive Bayes ═══
from sklearn.naive_bayes import MultinomialNB

# === SIZNING KODINGIZ (2 qator) ===
# 1. MultinomialNB(alpha=1.0) yarating -> clf_nb
# 2. clf_nb.fit(X_train, y_train) bilan o'qiting
clf_nb = MultinomialNB(alpha=1.0)
clf_nb.fit(X_train, y_train)


In [ ]:
# ─── Tekshirish ────────────────────────────────────────────────────────────
from sklearn.exceptions import NotFittedError
assert clf_nb is not None, "clf_nb None. MultinomialNB() yaratdingizmi?"
try:
    _acc_nb = clf_nb.score(X_train, y_train)
except NotFittedError:
    raise AssertionError("clf_nb o'qitilmagan. clf_nb.fit(X_train, y_train) ni chaqiring.")
assert set(clf_nb.classes_) == {"ijobiy", "salbiy"}, "Sinflar {'ijobiy','salbiy'} bo'lishi kerak."
assert _acc_nb > 0.8, f"O'quv aniqligi past ({_acc_nb:.2f})."
print(f"✓ Naive Bayes o'qitildi. O'quv aniqligi: {_acc_nb:.3f}")


### 4D. Namuna: Metrikalarni Confusion Matrix dan Hisoblaymiz — L2 [I3]-Slayd

Accuracy aldashi mumkin! Tengsiz sinflarda **F1** muhimroq.


In [ ]:
# ═══ NAMUNA: metrikalar — L2 [I3]-slayd (Model B) ═══
# TP=60, TN=820, FP=80, FN=40
TP, TN, FP, FN = 60, 820, 80, 40

accuracy  = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP)
recall    = TP / (TP + FN)
f1        = 2 * precision * recall / (precision + recall)

print(f"accuracy  = {accuracy:.3f}")
print(f"precision = {precision:.3f}")
print(f"recall    = {recall:.3f}")
print(f"F1        = {f1:.3f}")

# ─── ASSERT: Ma'ruza L2 [I3]-slayd bilan solishtiring ─────────────────────
assert abs(accuracy  - 0.88)   < 0.01
assert abs(precision - 0.4286) < 0.01
assert abs(recall    - 0.60)   < 0.01
assert abs(f1        - 0.50)   < 0.01
# Ma'ruza L2 [I3]-slayd bilan solishtiring
print("\n✓ Ma'ruza L2 [I3]-slayd tasdiqlandi: A=0.88, P≈0.43, R=0.60, F1=0.50")
print("  Diqqat: accuracy 0.88 yuqori, lekin F1 atigi 0.50 — tengsiz sinflar aldashi!")


### 4E. Birgalikda: Haqiqiy Modellarni Sinov To'plamida Baholaymiz


In [ ]:
# ═══ BIRGALIKDA: sinov to'plamida bashorat ═══
# === SIZNING KODINGIZ (2 qator) ===
# clf_lr va clf_nb bilan X_test uchun bashorat qiling
y_pred_lr = clf_lr.predict(X_test)
y_pred_nb = clf_nb.predict(X_test)


In [ ]:
# ─── Tekshirish ────────────────────────────────────────────────────────────
import numpy as np
assert y_pred_lr is not None and y_pred_nb is not None, (
    "Bashoratlar None. clf.predict(X_test) ni chaqiring."
)
assert len(y_pred_lr) == X_test.shape[0], "Bashorat uzunligi test hajmiga mos emas."
assert set(np.unique(y_pred_lr)) <= {"ijobiy", "salbiy"}, "Yorliqlar faqat ijobiy/salbiy."
print(f"✓ Bashoratlar tayyor: {len(y_pred_lr)} ta (LogReg), {len(y_pred_nb)} ta (NB)")


In [ ]:
# ─── (Berilgan) hisobot, confusion matrix va grafik ───────────────────────
from sklearn.metrics import classification_report, confusion_matrix, f1_score

for name, y_pred in [("LogReg", y_pred_lr), ("Naive Bayes", y_pred_nb)]:
    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred, digits=3, zero_division=0))
    print("Confusion matrix [ijobiy, salbiy]:")
    print(confusion_matrix(y_test, y_pred, labels=["ijobiy", "salbiy"]))

f1_lr = f1_score(y_test, y_pred_lr, pos_label="ijobiy")
f1_nb = f1_score(y_test, y_pred_nb, pos_label="ijobiy")
print(f"\nF1(ijobiy) — LogReg: {f1_lr:.3f}   Naive Bayes: {f1_nb:.3f}")
assert f1_lr > 0.6 and f1_nb > 0.6, "F1 juda past — modellar o'qitilganini tekshiring."

# Grafik (matplotlib bo'lsa) — Kaggle da ko'rinadi
try:
    import matplotlib.pyplot as plt
    from sklearn.metrics import ConfusionMatrixDisplay
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, labels=["ijobiy", "salbiy"])
    plt.title("LogReg — Confusion Matrix"); plt.show()
except Exception as ex:
    print(f"(grafik o'tkazib yuborildi: {type(ex).__name__})")


### 4F. Birgalikda: Xato Tahlili — Qaysi Sharhlar Noto'g'ri Tasniflandi?


In [ ]:
# ═══ BIRGALIKDA: xato tahlili ═══
# === SIZNING KODINGIZ (1-2 qator) ===
# y_test va y_pred_lr ni solishtirib, mos kelmagan indekslarni toping -> wrong_idx
wrong_idx = [i for i in range(len(y_test)) if y_test[i] != y_pred_lr[i]]

print(f"LogReg {len(wrong_idx)} ta sharhni noto'g'ri tasnifladi ({len(y_test)} dan):")
for i in wrong_idx[:5]:
    print(f"  haqiqiy={y_test[i]:7s} bashorat={y_pred_lr[i]:7s}  '{X_test_txt[i][:50]}'")


In [ ]:
# ─── Tekshirish ────────────────────────────────────────────────────────────
assert wrong_idx is not None, "wrong_idx None. Mos kelmagan indekslarni toping."
assert isinstance(wrong_idx, list), "wrong_idx ro'yxat bo'lishi kerak."
assert all(y_test[i] != y_pred_lr[i] for i in wrong_idx), "wrong_idx da to'g'ri tasniflangan element bor."
assert len(wrong_idx) < len(y_test), "Model hammasini xato qildi — qayta tekshiring."
print(f"✓ Xato tahlili: {len(wrong_idx)}/{len(y_test)} noto'g'ri tasniflandi (tasodifdan yaxshi).")


### 4G. Mustaqil: O'z Sharhlaringizni Tasniflang

Scaffold yo'q — yuqorida o'qitilgan `clf_lr` va `vec` dan foydalaning.


In [ ]:
# ═══ MUSTAQIL (Siz qilasiz): o'z sharhlaringizni tasniflang ═══
# 1. O'z sharhlaringizni yozing (yoki namunadan foydalaning)
# 2. to_clean_string() bilan tozalang
# 3. vec.transform() -> clf_lr.predict()
my_reviews = [
    "Bu mahsulot juda zo'r, hammaga tavsiya qilaman!",
    "Sifatsiz narsa keldi, pulim behuda ketdi, afsus.",
    "Yetkazib berish tez, lekin qadoqlash o'rtacha edi.",
]
my_clean = [to_clean_string(t) for t in my_reviews]
my_pred  = clf_lr.predict(vec.transform(my_clean))

for r, p in zip(my_reviews, my_pred):
    print(f"  [{p.upper():7s}]  {r}")


In [ ]:
# ─── Tekshirish ────────────────────────────────────────────────────────────
import numpy as np
assert my_pred is not None, "my_pred None. vec.transform + clf_lr.predict ni qo'llang."
assert len(my_pred) == len(my_reviews), "Bashorat soni sharhlar soniga mos emas."
assert set(np.unique(my_pred)) <= {"ijobiy", "salbiy"}, "Yorliqlar faqat ijobiy/salbiy."
print(f"✓ Mustaqil: {len(my_pred)} sharh tasniflandi -> {list(my_pred)}")


---
## §5  Loyihaga Ulash — m02 SentimentClassifier Yozamiz (~13 daqiqa)

4A-4F da o'rgangan g'oyalarni (TF-IDF + LR/NB + baholash) endi rasmiy
kapstone moduliga o'tkazamiz. Imzolar `capstone/contracts.py` da belgilangan —
o'zgartirmaymiz. Modul ichida m01 `TextPreprocessor` ishlatiladi.


In [ ]:
# ═══ §5: m02 SentimentClassifier modulini yozamiz ═══
from pathlib import Path
REPO_ROOT = Path().resolve().parent.parent
M02_PATH = REPO_ROOT / 'capstone' / 'modules' / 'm02_sentiment_classifier.py'
M02_PATH.parent.mkdir(parents=True, exist_ok=True)

M02_CODE = '''"""
capstone/modules/m02_sentiment_classifier.py
SentimentClassifier — TF-IDF + LogisticRegression / MultinomialNB asosida
ikkilik sentiment tahlili (ijobiy / salbiy).
Shartnoma: capstone/contracts.py :: SentimentClassifier
P2 (3-kun amaliyoti) da qurilgan; m01 (TextPreprocessor) ustiga quriladi.
Consumed by: M4 (FastAPI app.py), Day 16 (agent tool).
"""
from __future__ import annotations

import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

try:
    from m01_text_preprocessor import TextPreprocessor
except ImportError:  # paket sifatida import qilinganda
    from .m01_text_preprocessor import TextPreprocessor


class SentimentClassifier:
    """TF-IDF + LogReg yoki NaiveBayes asosida ikkilik sentiment tahlili.

    Consumed by: M4 (FastAPI), Day 16 (agent tool).
    """

    def __init__(self, model: str = "logreg") -> None:
        if model not in ("logreg", "nb"):
            raise ValueError("model 'logreg' yoki 'nb' bo'lishi kerak.")
        self._model_name = model
        self._pre = TextPreprocessor()
        self._vec = TfidfVectorizer()
        self._clf = (
            LogisticRegression(max_iter=1000)
            if model == "logreg"
            else MultinomialNB(alpha=1.0)
        )
        self._fitted = False

    def _prep(self, text: str) -> str:
        """m01 bilan tozalab, TF-IDF uchun bo'sh-joy bilan birlashtiradi."""
        if not isinstance(text, str) or not text.strip():
            return ""
        return " ".join(self._pre.preprocess(text))

    def fit(self, texts: list[str], labels: list[str]) -> None:
        """Modelni o'qitadi. labels: 'ijobiy' yoki 'salbiy'."""
        if len(texts) != len(labels):
            raise ValueError("texts va labels uzunligi teng bo'lishi kerak.")
        X = self._vec.fit_transform([self._prep(t) for t in texts])
        self._clf.fit(X, labels)
        self._fitted = True

    def predict(self, text: str) -> str:
        """Bitta matn uchun 'ijobiy' yoki 'salbiy' qaytaradi."""
        if not self._fitted:
            raise ValueError("Avval fit() ni chaqiring.")
        X = self._vec.transform([self._prep(text)])
        return str(self._clf.predict(X)[0])

    def predict_proba(self, text: str) -> dict[str, float]:
        """Ehtimolliklar: {'ijobiy': 0.82, 'salbiy': 0.18}."""
        if not self._fitted:
            raise ValueError("Avval fit() ni chaqiring.")
        X = self._vec.transform([self._prep(text)])
        probs = self._clf.predict_proba(X)[0]
        return {str(c): float(p) for c, p in zip(self._clf.classes_, probs)}

    def save(self, path: str) -> None:
        """Vektorlashtiruvchi va modelni pickle orqali saqlaydi."""
        with open(path, "wb") as f:
            pickle.dump(
                {
                    "vec": self._vec,
                    "clf": self._clf,
                    "model_name": self._model_name,
                    "fitted": self._fitted,
                },
                f,
            )

    def load(self, path: str) -> None:
        """Saqlangan modelni yuklaydi."""
        with open(path, "rb") as f:
            state = pickle.load(f)
        self._vec = state["vec"]
        self._clf = state["clf"]
        self._model_name = state["model_name"]
        self._fitted = state["fitted"]
'''

M02_PATH.write_text(M02_CODE, encoding='utf-8')
print(f'OK: {M02_PATH} yozildi ({len(M02_CODE)} belgi).')


In [ ]:
# ─── m02 ni import qilib, shartnomaga muvofiqligini tekshiramiz ──────────────
import sys, os, tempfile
if str(MODULES_DIR) not in sys.path:
    sys.path.insert(0, str(MODULES_DIR))

# Yangi yozilgan modulni qayta yuklash (agar oldin import qilingan bo'lsa)
import importlib
import m02_sentiment_classifier
importlib.reload(m02_sentiment_classifier)
from m02_sentiment_classifier import SentimentClassifier

clf = SentimentClassifier(model="logreg")
clf.fit(texts, labels)

# Shartnoma: predict -> str
p_pos = clf.predict("Mahsulot juda sifatli, tez yetkazib berishdi, rahmat!")
p_neg = clf.predict("Sifatsiz narsa keldi, sindi, pulim behuda ketdi, afsus.")
assert p_pos == "ijobiy", f"Ijobiy sharh '{p_pos}' deb tasniflandi."
assert p_neg == "salbiy", f"Salbiy sharh '{p_neg}' deb tasniflandi."

# Shartnoma: predict_proba -> dict[str,float], yig'indi=1
pr = clf.predict_proba("Mahsulot zo'r va sifatli")
assert set(pr.keys()) == {"ijobiy", "salbiy"}, "predict_proba kalitlari noto'g'ri."
assert abs(sum(pr.values()) - 1.0) < 1e-6, "Ehtimolliklar yig'indisi 1 emas."

# Shartnoma: save/load roundtrip
tmp = os.path.join(tempfile.gettempdir(), "m02_test.pkl")
clf.save(tmp)
clf2 = SentimentClassifier()
clf2.load(tmp)
_t = "Mahsulot juda sifatli, rahmat!"
assert clf2.predict(_t) == clf.predict(_t), "save/load dan keyin bashorat mos kelmadi."

print("✓ m02 SentimentClassifier barcha shartnoma tekshiruvlaridan o'tdi!")
print(f"  predict(ijobiy sharh) = {p_pos}   predict(salbiy sharh) = {p_neg}")
print(f"  predict_proba = {{k: round(v,3) for k,v in pr.items()}}")


### 5C. Git — m02 ni Commit Qiling

```bash
git add capstone/modules/m02_sentiment_classifier.py
git commit -m "day03: capstone - m02 SentimentClassifier"
git push origin HEAD
```


In [ ]:
import subprocess
res = subprocess.run(
    ["git", "add", "capstone/modules/m02_sentiment_classifier.py"],
    capture_output=True, text=True, cwd=str(REPO_ROOT))
print("git add:", "OK" if res.returncode == 0 else res.stderr.strip())

st = subprocess.run(["git", "diff", "--cached", "--name-only"],
                    capture_output=True, text=True, cwd=str(REPO_ROOT))
if st.stdout.strip():
    cm = subprocess.run(
        ["git", "commit", "-m", "day03: capstone - m02 SentimentClassifier"],
        capture_output=True, text=True, cwd=str(REPO_ROOT))
    print(cm.stdout.strip() or "Commit bajarildi.")
else:
    print("Commit qilish uchun yangi o'zgarish yo'q.")


---
## §6  Tadqiqot Savoli + Yakun (~7 daqiqa)

**Savol:** Kichik o'zbek korpusida (bir necha yuz sharh) LogReg yoki Naive Bayes —
qaysi biri yaxshiroq ishlaydi? Nima uchun? (L2 [O]-slayd muhokamasi.)


In [ ]:
# Mini tadqiqot: LogReg vs Naive Bayes (sinov F1 si)
from sklearn.metrics import f1_score

f1_lr = f1_score(y_test, clf_lr.predict(X_test), pos_label="ijobiy")
f1_nb = f1_score(y_test, clf_nb.predict(X_test), pos_label="ijobiy")

print(f"LogReg      F1(ijobiy) = {f1_lr:.3f}")
print(f"Naive Bayes F1(ijobiy) = {f1_nb:.3f}")
print(f"Farq                   = {abs(f1_lr - f1_nb):.3f}")
print()
print("Mulohaza: kichik korpusda NB ko'pincha barqaror (kam parametr),")
print("LR esa ma'lumot ko'paygach yaxshilanadi. Aniq xulosa uchun")
print("kross-validatsiya (cross_val_score) bilan bir necha bo'linmada sinang.")


---
## Yakun

**Bugun nimalar qildik:**
- ✓ Uzum Market sharhlarini ijobiy/salbiy ga binarizatsiya qildik (reyting 3 tashlandi)
- ✓ m01 (P1) bilan tozalab, TF-IDF + train/test ajratdik
- ✓ Naive Bayes ni qo'lda hisoblab, L2 [I2]-slayd natijasini tasdiqladik (nisbat=3.375)
- ✓ LogisticRegression va MultinomialNB ni o'qitdik
- ✓ precision/recall/F1 va confusion matrix bilan baholadik (L2 [I3]: F1=0.50)
- ✓ m02 `SentimentClassifier` ni capstone moduliga yozdik

**Ertaga (P3 — So'z embeddinglari):**
m01 -> tayyor embeddinglar (cc_uz_100k.kv) -> kosinus o'xshashlik -> m03 PretrainedEmbedder
Birinchi assert: `abs(cos_val - 0.667) < 1e-3` (L3 [I2]-slayd)

---

> **Chiqish chiptasi:** Bugun eng tushunarsiz qolgan narsa nima?
> (Quyidagi katakka yozing — keyingi darsda muhokama qilamiz.)
